In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [2]:
parsed_args = RayTracing.parse_commandline()
    
# set up logging
logger = RayTracing.setup_logging(parsed_args["debug"])
RayTracing.global_logger(logger)

# set random seed
RayTracing.Random.seed!(parsed_args["seed"])

BASE_FNAME = "hehe.exr"

parsed_args["image-dim"] = [150, 150]
parsed_args["samples-per-pixel"] = 16

16

In [3]:
function unit_circle_points(N::Int, x_shift::Real=0.0, y_shift::Real=0.0, x_scale::Real=1.0, y_scale::Real=1.0)
    # Check for valid input
    if N <= 0
        throw(ArgumentError("Number of points must be positive"))
    end
    
    # Calculate angular spacing in radians (2π divided by N)
    θ_step = 2π / N
    
    # Generate the points
    points = Vector{Tuple{Float64, Float64}}(undef, N)
    
    for i in 1:N
        # Calculate the angle for this point
        θ = (i - 1) * θ_step
        
        # Convert to Cartesian coordinates, apply scaling and shifting
        x = cos(θ) * x_scale + x_shift
        y = sin(θ) * y_scale + y_shift
        
        # Store the point
        points[i] = (x, y)
    end
    
    return points
end

unit_circle_points (generic function with 5 methods)

In [4]:
unit_circle_points(16, 0.0, 1.9, .5, .5)

16-element Vector{Tuple{Float64, Float64}}:
 (0.5, 1.9)
 (0.46193976625564337, 2.0913417161825447)
 (0.3535533905932738, 2.2535533905932734)
 (0.19134171618254492, 2.3619397662556434)
 (3.061616997868383e-17, 2.4)
 (-0.19134171618254486, 2.3619397662556434)
 (-0.35355339059327373, 2.253553390593274)
 (-0.46193976625564337, 2.0913417161825447)
 (-0.5, 1.9)
 (-0.4619397662556434, 1.708658283817455)
 (-0.35355339059327384, 1.5464466094067262)
 (-0.19134171618254517, 1.4380602337443567)
 (-9.184850993605148e-17, 1.4)
 (0.191341716182545, 1.4380602337443567)
 (0.3535533905932737, 1.546446609406726)
 (0.46193976625564326, 1.7086582838174547)

In [10]:
# manually build scene
for (i, (shear_c, scale_y)) in enumerate(unit_circle_points(16, 0.0, 1.9, .5, .5))
    parsed_args["file-name"] = replace(BASE_FNAME, ".exr" => "_$(lpad(string(i),2,"0")).exr")
    primitives = RayTracing.Primitive[]
    lights = RayTracing.Light[]

    old_smile = RayTracing.jmfp("/Users/johnmyslinski/Documents/PBRJ/ref/smile3.png")
    new_smile = RayTracing.jmfp("/Users/johnmyslinski/Documents/PBRJ/ref/smile3_post.exr")

    # make my PNG blue!
    RayTracing.party_blob_fuckery!(
        old_smile,
        new_smile,
        (0.3, 0.3, 1.3)
    )

    # materials
    mat_gray = RayTracing.Matte(
        RayTracing.ConstantTexture(RayTracing.spectrum_from_float(0.5, 0.5, 0.5)),
        RayTracing.ConstantTexture(RayTracing.spectrum_from_float(0.0, 0.0, 0.0)),
        nothing
    )

    emissive_color = RayTracing.spectrum_from_float(0.3, 0.3, 1.3)
    Kd = RayTracing.MixMultTexture(
        RayTracing.ConstantTexture(emissive_color),
        RayTracing.ImageTexture(RayTracing.UVMapping2D(), new_smile)
    )
    mat_blob = RayTracing.Matte(
        Kd,
        RayTracing.ConstantTexture(RayTracing.spectrum_from_float(0.0, 0.0, 0.0)),
        nothing
    )

    ###############
    ### a thing ###
    ###############       

    radius = 1.0
    sphere_t = RayTracing.Shear(0.0, 0.0, shear_c, 0.0, 0.0, 0.0) * RayTracing.Scale(1.0, scale_y, 1.0) * RayTracing.RotateY(130.0) * RayTracing.RotateZ(-35.0) * RayTracing.RotateX(-80.0)
    sphere = RayTracing.Sphere(
        RayTracing.ShapeCore(sphere_t, RayTracing.Inv(sphere_t), false, false),
        radius
    )
    alight = RayTracing.DiffuseAreaLight(
        emissive_color,
        sphere,
        false,
        nothing,
        new_smile,
        1.0
    )
    push!(primitives, RayTracing.Primitive(sphere, mat_blob, alight))
    push!(lights, alight)

    floor_transform = RayTracing.Translate(RayTracing.Pnt3(0,0,0))
    floor = RayTracing.Rectangle(
        RayTracing.Pnt2(-10, -10),
        RayTracing.Pnt2(10, 10),
        0.0,
        2, 
        RayTracing.ShapeCore(floor_transform, RayTracing.Inv(floor_transform), false, false),
        false,
        nothing
    )
    for tri in floor
        push!(primitives, RayTracing.Primitive(tri, mat_gray, nothing))
    end

    # instantiate accelerator
    print("\nThere are " * RayTracing.num2str(length(primitives)) * " objects in the scene, building BVH\n")
    @time bvh = RayTracing.BVH(primitives)
    print("Done building BVH\n")

    # Instantiate a Filter
    filter = RayTracing.BoxFilter(RayTracing.Pnt2(.5, .5))

    # Instantiate a Film
    film = RayTracing.Film(
        RayTracing.Pnt2(parsed_args["image-dim"][1], parsed_args["image-dim"][2]),
        RayTracing.Bounds2(RayTracing.Pnt2(parsed_args["crop-window"][1], parsed_args["crop-window"][2]), RayTracing.Pnt2(parsed_args["crop-window"][3], parsed_args["crop-window"][4])),
        filter,
        1.0,
        1.0,
        parsed_args["file-name"]
    )

    # Instantiate a Camera
    look_from = RayTracing.Pnt3(4, 4, 4)
    look_at = RayTracing.Pnt3(0, radius, 0)
    up = RayTracing.Vec3(0, 1, 0)
    screen = RayTracing.Bounds2(RayTracing.Pnt2(-1, -1), RayTracing.Pnt2(1, 1))
    C = RayTracing.PerspectiveCamera(RayTracing.LookAt(look_from, look_at, up), screen, 0.0, 1.0, 0.0, 1e6, 37.0, film)

    # Instantiate a Sampler
    S = RayTracing.ZSobolSampler(
        parsed_args["samples-per-pixel"], 
        RayTracing.Pnt2(parsed_args["image-dim"][1], parsed_args["image-dim"][2]), 
        Int8(2)
    )
    print("Using " * RayTracing.num2str(S.samples_per_pixel) * " samples per pixel\n")
    
    # Instantiate Scene
    print("There are " * RayTracing.num2str(length(lights)) * " lights in the scene\n")
    scene = RayTracing.Scene(lights, bvh)
    
    # Instantiate an Integrator
    I = RayTracing.BDPTIntegrator(C, S, parsed_args["max-depth"])

    image = RayTracing.render(
        I, 
        scene, 
        parsed_args,
        (-1, -1)
    )
    RayTracing.OpenEXR.save(I.camera.core.core.film.filename, image)
end




There are 3 objects in the scene, building BVH
  0.000008 seconds (16 allocations: 1.188 KiB)
Done building BVH
Using 16 samples per pixel
There are 1 lights in the scene
Rendering 100 tiles
Utilizing 1 threads



Progress: 100%|█████████████████████████████████████████| Time: 0:00:15



There are 3 objects in the scene, building BVH
  0.000010 seconds (16 allocations: 1.188 KiB)
Done building BVH
Using 16 samples per pixel
There are 1 lights in the scene
Rendering 100 tiles
Utilizing 1 threads



Progress: 100%|█████████████████████████████████████████| Time: 0:00:15



There are 3 objects in the scene, building BVH
  0.000010 seconds (16 allocations: 1.188 KiB)
Done building BVH
Using 16 samples per pixel
There are 1 lights in the scene
Rendering 100 tiles
Utilizing 1 threads



...

Progress: 100%|█████████████████████████████████████████| Time: 0:00:14



There are 3 objects in the scene, building BVH
  0.000011 seconds (16 allocations: 1.188 KiB)
Done building BVH
Using 16 samples per pixel
There are 1 lights in the scene
Rendering 100 tiles
Utilizing 1 threads



Progress: 100%|█████████████████████████████████████████| Time: 0:00:14



There are 3 objects in the scene, building BVH
  0.000010 seconds (16 allocations: 1.188 KiB)
Done building BVH
Using 16 samples per pixel
There are 1 lights in the scene
Rendering 100 tiles
Utilizing 1 threads



Progress: 100%|█████████████████████████████████████████| Time: 0:00:15


In [11]:
# then run this to convery to gif
# convert -delay 1 -loop 0 -set colorspace RGB -colorspace sRGB *.exr barty-plob.gif